# Experimento 3 — SVM sem balanceamento utilizando todas as variáveis

Este experimento expande o conjunto de variáveis utilizadas pelo modelo SVM. Além das quatro variáveis iniciais, agora são incluídos parâmetros físico-químicos diretamente relacionados à qualidade da água.

O objetivo é avaliar se o acréscimo de mais informações ambientais melhora a capacidade do modelo em separar as classes do `conama_status`, especialmente as classes minoritárias `Atenção` e `Crítica`.

Diferente do Experimento 2, este experimento não utiliza balanceamento de classes. Assim, o objetivo é isolar o impacto da ampliação das variáveis de entrada.

As variáveis utilizadas foram:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `pH (ph units)`
- `Biochemical Oxygen Demand (mg/l)`
- `Dissolved Oxygen (mg/l)`
- `Ammonia (mg/l)`
- `Country`
- `Waterbody Type`

A variável alvo (`y`) foi:

- `conama_status`

O dataset foi dividido em:

- 80% para treino
- 20% para teste

Foi utilizado o `LinearSVC`, pois o `SVC` tradicional apresentou custo computacional elevado para o tamanho do dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

print("Bibliotecas carregadas.")

Bibliotecas carregadas.


In [4]:
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path(
        "C:/Users/anaju\OneDrive/projetos_lucas/AquaSense-ML/amostra_rotulada.parquet"
    )

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente local/VS Code detectado.
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("Bibliotecas de Machine Learning carregadas.")

Bibliotecas de Machine Learning carregadas.


In [6]:
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "pH (ph units)",
        "Biochemical Oxygen Demand (mg/l)",
        "Dissolved Oxygen (mg/l)",
        "Ammonia (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

print("X e y definidos.")
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

X e y definidos.
Shape de X: (141399, 8)
Shape de y: (141399,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 8)
Teste: (28280, 8)


In [8]:
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Ammonia (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

print("Pré-processamento configurado.")

Pré-processamento configurado.


In [9]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LinearSVC(
                random_state=SEED,
                max_iter=5000
            )
        )
    ]
)

print("Iniciando treinamento do SVM sem balanceamento com todas as variáveis...")

model.fit(X_train, y_train)

print("SVM treinado com sucesso.")

Iniciando treinamento do SVM sem balanceamento com todas as variáveis...
SVM treinado com sucesso.


## Avaliação das Métricas de Treino

Nesta etapa, avaliamos o desempenho do modelo no conjunto de treino para observar se há indícios de overfitting ou underfitting.

In [10]:
print("Calculando métricas de treino...")

y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("\nTrain Precision:")
print(train_precision)

print("\nTrain Recall:")
print(train_recall)

print("\nTrain F1:")
print(train_f1)

print("\nTrain Confusion Matrix:")
print(train_confusion_matrix)

Calculando métricas de treino...
Train Accuracy:
0.8288085997931381

Train Precision:
0.7980490335300839

Train Recall:
0.8288085997931381

Train F1:
0.8029417116673655

Train Confusion Matrix:
[[ 3209  3843     0   508]
 [ 1048  8371     0 12259]
 [  431   660     0    15]
 [    0   601     0 82174]]


## Avaliação no Conjunto de Teste

Nesta etapa, avaliamos o modelo em dados não vistos durante o treinamento. A análise considera accuracy, precision, recall, F1-score e matriz de confusão.

In [11]:
print("Calculando métricas de teste...")

y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

test_precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:")
print(test_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Calculando métricas de teste...
Accuracy:
0.8288189533239039

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.68      0.41      0.51      1890
         Boa       0.62      0.39      0.48      5419
     Crítica       0.00      0.00      0.00       277
   Excelente       0.87      0.99      0.92     20694

    accuracy                           0.83     28280
   macro avg       0.54      0.45      0.48     28280
weighted avg       0.80      0.83      0.80     28280


Confusion Matrix:
[[  780   970     0   140]
 [  251  2116     0  3052]
 [  109   166     0     2]
 [    0   151     0 20543]]


## Resultados Obtidos — Experimento 3

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.829
```

O Experimento 3 apresentou melhora significativa na acurácia em comparação aos experimentos anteriores com SVM. A principal modificação realizada foi a ampliação do conjunto de variáveis ambientais utilizadas pelo modelo.

Além das variáveis originalmente utilizadas (`Temperature`, `Orthophosphate`, `Country` e `Waterbody Type`), foram adicionados novos parâmetros físico-químicos relacionados diretamente à qualidade da água, como:

- pH;
- Biochemical Oxygen Demand;
- Dissolved Oxygen;
- Ammonia.

O objetivo foi verificar se a inclusão dessas variáveis aumentaria a capacidade de separação entre as classes do `conama_status`.



## Comparação entre algoritmos — Experimento 3

| Métrica | RF Exp. 3 | LightGBM Exp. 3 | Regressão Logística Exp. 3 | SVM Exp. 3 |
|---|---:|---:|---:|---:|
| Accuracy teste | 0.821 | 0.791 | 0.801 | 0.829 |
| Precision Atenção | 0.63 | 0.58 | 0.61 | 0.68 |
| Precision Boa | 0.59 | 0.55 | 0.57 | 0.62 |
| Precision Crítica | 0.00 | 0.00 | 0.00 | 0.00 |
| Precision Excelente | 0.86 | 0.84 | 0.85 | 0.87 |
| Recall Atenção | 0.37 | 0.31 | 0.35 | 0.41 |
| Recall Boa | 0.34 | 0.28 | 0.31 | 0.39 |
| Recall Crítica | 0.00 | 0.00 | 0.00 | 0.00 |
| Recall Excelente | 0.98 | 0.97 | 0.98 | 0.99 |
| F1 Atenção | 0.47 | 0.40 | 0.45 | 0.51 |
| F1 Boa | 0.43 | 0.37 | 0.41 | 0.48 |
| F1 Crítica | 0.00 | 0.00 | 0.00 | 0.00 |
| F1 Excelente | 0.91 | 0.90 | 0.91 | 0.92 |
| Macro avg F1 | 0.45 | 0.42 | 0.44 | 0.48 |
| Weighted avg F1 | 0.78 | 0.75 | 0.77 | 0.80 |



## Análise das métricas

O SVM do Experimento 3 apresentou o melhor desempenho geral entre todos os algoritmos testados nesse cenário utilizando ampliação das variáveis ambientais.

A acurácia atingiu aproximadamente 0.829, superando:
- Random Forest (0.821);
- Regressão Logística (0.801);
- LightGBM (0.791).

Além da maior acurácia, o SVM também apresentou:
- maior weighted F1-score;
- melhor precision para `Atenção`;
- melhor precision para `Boa`;
- melhor recall para `Atenção`;
- melhor recall para `Boa`.

Esses resultados indicam que o SVM conseguiu utilizar de forma mais eficiente as novas variáveis físico-químicas adicionadas ao dataset.

A melhora foi especialmente importante nas classes intermediárias `Atenção` e `Boa`, que historicamente apresentavam forte confusão com a classe `Excelente`.

O F1-score da classe `Boa` atingiu 0.48, representando avanço significativo em relação aos experimentos anteriores e também desempenho superior aos demais algoritmos avaliados nesse experimento.

A classe `Excelente` manteve desempenho extremamente alto em todos os modelos, mas o SVM novamente apresentou o melhor resultado, com:
- recall de 0.99;
- F1-score de 0.92.

Entretanto, todos os algoritmos continuaram apresentando dificuldade extrema na classe `Crítica`.

Assim como ocorreu nos experimentos dos demais integrantes do grupo, nenhuma abordagem conseguiu identificar corretamente essa classe sem utilização explícita de técnicas de balanceamento.

Isso mostra que:
- o problema principal deixou de ser falta de variáveis;
- e passou a ser o desbalanceamento extremo da distribuição das classes.



## Análise da Matriz de Confusão

A matriz de confusão mostra melhora significativa na distribuição das previsões do SVM.

O modelo conseguiu identificar corretamente:

- 20543 amostras da classe `Excelente`;
- 2116 amostras da classe `Boa`;
- 780 amostras da classe `Atenção`.

Em comparação aos experimentos anteriores, houve redução importante da confusão entre:
- `Boa` e `Excelente`;
- `Atenção` e `Excelente`.

Isso demonstra que as novas variáveis ambientais forneceram maior capacidade de separação entre as classes.

Entretanto, as amostras da classe `Crítica` continuaram sendo classificadas principalmente como `Boa` e `Atenção`, evidenciando que o modelo ainda sofre influência do forte desbalanceamento do dataset.



## Conclusão

O Experimento 3 demonstrou que a ampliação das variáveis ambientais teve impacto extremamente positivo no desempenho do SVM.

Entre todos os algoritmos comparados neste experimento, o SVM apresentou:
- maior acurácia;
- maior weighted F1-score;
- melhor desempenho geral nas classes intermediárias;
- melhor equilíbrio entre precision e recall.

Os resultados sugerem que o SVM conseguiu aproveitar melhor os parâmetros físico-químicos adicionais para separar as classes do `conama_status`.

Entretanto, o problema da classe `Crítica` permaneceu sem solução em todos os modelos avaliados. Isso indica que:
- aumentar variáveis melhora significativamente o desempenho geral;
- mas não resolve sozinho o problema do desbalanceamento extremo.

Portanto, os próximos experimentos do projeto AquaSense podem combinar:
- ampliação das variáveis ambientais;
- técnicas de balanceamento;
- ajuste de hiperparâmetros;
- métodos mais robustos para classes raras.

Essa combinação pode melhorar simultaneamente:
- desempenho global;
- estabilidade das previsões;
- detecção de amostras críticas ambientalmente relevantes.